# Claude Platform：ハンズオン

**レベル：** 300（中級） | **モデル：** Claude Sonnet 5（`claude-sonnet-5`）

## 本演習で構築するもの

チケットの詳細を読み込み、ナレッジベースを検索し、構造化された解決結果を出力する、マルチツールの**サポートチケットエージェント**です。フレームワークは使用せず、Claude APIを直接呼び出します。

**お客様シナリオ：** 中堅B2B SaaS企業のTechFlow（1日に500件以上のチケット）は、Tier 1の一次切り分けを自動化し、担当者が複雑なエスカレーション案件に集中できる体制を目指しています。

## 学習目標

1. マルチツールのエージェントループ（`while stop_reason == "tool_use"`）を実装する
2. 1つのエージェントの中で、構造化出力とツール使用を組み合わせる
3. アダプティブ思考と `output_config.effort` を組み合わせ、推論の深さを制御する
4. 思考、ツール呼び出し、応答をリアルタイムでストリーミングする

## 本セッションの位置づけ

Claudeを活用した開発には、いくつかの方法があります。本セッションでは、最も下位のレイヤーにあたる**Messages API**を扱います。リクエストとレスポンスのやり取りを完全に制御できる方法です。

| 開発手段 | 概要 | 主な用途 |
|---------|-----------|----------------|
| **Messages API** ← *本セッション* | Claudeへの直接のHTTP／SDK呼び出し | エージェントループの完全な制御、独自のオーケストレーション、本番バックエンド |
| **Agent SDK** | ツールのディスパッチを組み込んだPythonフレームワーク | エージェントの迅速なプロトタイピング、オーケストレーションをフレームワークに任せたい場合 |
| **Claude Code** | CLIベースのコーディングエージェント | 開発者の生産性向上、リポジトリ単位のタスク、対話型のコーディング |
| **claude.ai / Claude for Enterprise** | プロジェクトやMCPを備えたチャットインターフェース | エンドユーザーの業務フロー、企業のナレッジワーク、開発者以外の用途 |

本日は、基盤となるループを一から実装します。この仕組みを理解すると、上位のレイヤーもすべて理解しやすくなります。

---

**Partner Basecamp参加者向けの資料です。** 研修資料としての複製・再配布はご遠慮ください。ここで扱うパターンは、皆さまのお客様案件で自由にご活用いただけます。

## セットアップ

以下のセルを実行し、依存パッケージのインストール、APIキーの設定、モックデータの読み込みを行ってください。

- **APIキー**は、セットアップセルが作成する `.env` ファイルに貼り付けます。貼り付けは初回のみで済みます（gitignoreの対象であり、カーネルを再起動しても保持されます）。
- 本日は、**✏️ 実装課題のセル**にコードを記述していただきます。
- **APIを呼び出すセルには時間がかかります。** 思考を伴うエージェントの実行には、30〜90秒かかる場合があります。セル横の `[*]` は、処理中であることを示します。


### セットアップ：Claudeへの接続

最初に次のセルを実行してください。セットアップセルは、初回実行時に **`.env` ファイル**を作成します（gitignoreの対象のため、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=` の後ろにキーを貼り付けて保存し、セルを再実行してください。キーはカーネルを再起動しても保持されるため、貼り付けは1回で済みます。*（`.env` がまだない場合は、代わりに、入力した文字が表示されない入力欄でキーの入力を求められます。）* 緑色の **「✓ APIキーを確認しました」** バナーが表示されれば、接続は完了です。赤いバナーが表示された場合は、表示された指示に従ってセルを再実行してください。

In [ ]:
# ── パッケージのインストールとインポート ──
# このカーネルに依存パッケージをインストール。再実行しても安全で、制限付きのPython（PEP 668）でも動作する
import importlib.util, os, subprocess, sys

# ── 環境ガード ────────────────────────────────────────────────────────────
# 下のインストール処理と同じセルに置き、スキップできないようにしている。これにより、
# 素のシステムPythonにパッケージがインストールされることはない（従来の最悪のケースは、
# IT部門が管理するマシンでの --break-system-packages の実行）。ここで停止した場合は、
# SETUP.mdの「ノートブックが停止する理由」を参照。
def _in_isolated_env():
    """実行中のカーネルがvenv／virtualenv／conda環境、またはColabであればTrueを返す。
    判定はインタープリター自身（sys.*）に基づく。起動元のシェルから引き継いだ
    有効化用の環境変数は、sys.executableが実際にその環境の内部にある場合に
    限り信頼する。有効化済みのターミナルから起動したシステムPythonの
    カーネルもVIRTUAL_ENVを引き継ぐが、この場合は
    判定を通してはならない。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨ての独自ランタイムでサンドボックス化されている
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405のvenv（python -m venv）。大半のconda環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 従来のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルから引き継いだ情報にすぎず、カーネルは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有 `base` は分離環境とみなさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードを回避しました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変更されます。\n"
        "会社が管理するノートPCでは、IT部門への申請が必要になる場合があります。\n"
        "\n"
        "対処方法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行してください：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windowsの場合：.venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、右上のカーネル名をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が.venvの環境を選択 → このセルを再実行してください。\n"
        "\n"
        "condaをご利用の場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選択してください。\n"
        "ご自身で管理するマシンを使うファシリテーターは、BASECAMP_ALLOW_SYSTEM_PYTHON=1 で回避できます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表示（緑のボックス、エラー時は赤）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードによりこのセルを停止しました。上のメッセージをご確認ください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：分離されたインタープリターを検出しました。安全にインストールできます")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: (import_name, pip_spec) のリスト。不足しているものだけを、
    実行中のインタープリターにインストールする。通常のインストール、ユーザー領域、
    PEP 668の上書き（ユーザー領域を優先し、システム全体は最後の手段）の順に試行する。
    いずれの試行も出力を表示しない（pipの出力はストリーム表示せずに取り込む）。そのため、
    制限付きのPython（HomebrewやDebian、PEP 668）でフォールバックが実際に成功した場合に、
    'externally-managed-environment' の長いエラー文が表示されることはない。
    すべての方法が失敗した場合に限り、生のトレースバックではなく、
    venvによる対処方法とともに原因を表示する。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + " — first run only, please wait…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipからの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonは制限付き（PEP 668）か、オフラインの状態です。最も手早い対処方法はvenvの利用です：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合はSETUP.mdを参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネル選択）で.venvのインタープリターを選択し、Run Allを実行してください。\n"
        "  社内プロキシやPyPIのブロックが原因の場合は、リポジトリのルートにあるSETUP.mdをご参照ください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ 依存パッケージの準備が完了しました")

import anthropic
import json
import time
import os

# ── APIキーの設定 ──
import os

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキストで表示する。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("not in a notebook kernel - use the plain-text banner")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeへの接続：Anthropic APIまたはAmazon Bedrock ──
# どちらの認証情報でも動作する。種類はこのセルが自動で判別する
#   Anthropic API：ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock：AWS_BEARER_TOKEN_BEDROCK=...  と  AWS_REGION=us-east-1
# 使用する認証情報を、このセルが作成する.envファイル（gitignoreの対象のため、コミットされない）に記入するか、
# シェルでexportする。シェルの値は.envファイルの値より優先される。
_ENV_TEMPLATE = (
    "# Anthropic APIキー：=の後ろに貼り付けてください（引用符・スペースは不要）。保存後、\n"
    "# セットアップのセルを再実行してください。キーは https://console.anthropic.com/ で取得できます。\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Amazon Bedrockを使用する場合は、上の行をコメントアウトし、次の項目を記入してください：\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上位へたどり、最も近い既存の.envを探す（ルートに.envを1つ置けば
    すべての演習で共用できる）。まだ存在しない場合はリポジトリのルートを指し、
    ノートブックを単独で開いた場合はこのフォルダーを指す。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 簡易的な.envパーサー（python-dotenvに依存しない）。実行のたびに読み直すため、キーを貼り付けて
# 再実行すれば反映される。環境（シェル／Claude Code／CI）に実際の値がある場合は
# そちらが優先され、プレースホルダーが残ることはない。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """黄色の警告バナーを表示して停止する。数セル先ではなく、この時点でセットアップを失敗させるため。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報が設定されていません。上のメッセージをご確認ください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIをご利用の場合は、次を設定してください：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockをご利用の場合は、次の両方を設定してください：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化しているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンの指定が必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効化しているリージョンを追記してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには `anthropic.` というプレフィックスが付きます。Anthropic APIでは、プレフィックスなしのIDを使用します。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使用するモデル。利用中のプロバイダーに合わせてIDを解決する
MODEL = _model("claude-sonnet-5")        # この演習の主力モデル
FAST_MODEL = _model("claude-haiku-4-5")  # 低コストかつ高速（接続確認、ジャッジ用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデルを試す場合


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続確認：認証情報に加え、このモデルを実際に利用できるかも検証する
# Bedrockでは、キーが有効でも、アカウント／リージョンでモデルが有効化されていなければ404になることがある。
# そのため、認証情報の形式を確認するだけでなく、実際のモデルIDにpingを送る。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージをご確認ください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "Bedrockキーが拒否されました。AWS_BEARER_TOKEN_BEDROCKの値と、"
                       "Bedrockの呼び出し権限が付与されていることを確認してから、このセルを再実行してください。")
    else:
        _status(False, "キーが拒否されました。このセルを再実行し、キー全体を貼り付けてください"
                       "（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルを再実行して、もう一度お試しください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "ネットワーク接続を確認してから、このセルを再実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや !python がこの値を参照する
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 実際に使用するクライアント。非ストリーミング呼び出しで max_tokens>21333 を使うには、長めのタイムアウトが必要
client = _make_client(timeout=900.0)


In [ ]:
# ── サンプルチケットのデータ ──

TICKETS = {
    "TKT-1042": {
        "id": "TKT-1042", "customer": "Acme Corp", "priority": "high",
        "product_area": "billing",
        "description": "3月分の請求書で二重に請求されています。請求書 #INV-2024-0342 の金額は$4,500ですが、銀行の明細には3月3日付で同額の請求が2件記録されています。重複分を至急ご返金ください。",
        "status": "open"
    },
    "TKT-1043": {
        "id": "TKT-1043", "customer": "DataFlow Inc", "priority": "medium",
        "product_area": "api",
        "description": "昨日APIキーをローテーションして以降、Webhookのエンドポイントがイベントを受信しなくなりました。新しいキーがREST呼び出しで使用できることは確認済みですが、Webhookは引き続き失敗しています。Webhookのログには401エラーが記録されています。",
        "status": "open"
    },
    "TKT-1044": {
        "id": "TKT-1044", "customer": "CloudScale Ltd", "priority": "low",
        "product_area": "feature_request",
        "description": "ダッシュボードへの一括エクスポート機能の追加を希望します。現在はレポートを1件ずつエクスポートする必要があり、50件以上のプロジェクトにまたがる四半期サマリーを作成する際に大きな負担となっています。",
        "status": "open"
    },
    "TKT-1045": {
        "id": "TKT-1045", "customer": "SecureNet Systems", "priority": "critical",
        "product_area": "account",
        "description": "MFA認証に繰り返し失敗したため、管理者アカウント（admin@securenet.io）がロックアウトされました。SSOがこの管理者アカウントに紐づいているため、47名のチームメンバーがプラットフォームにアクセスできず、すべての業務が停止しています。",
        "status": "open"
    },
    "TKT-1046": {
        "id": "TKT-1046", "customer": "MedTech Solutions", "priority": "high",
        "product_area": "api",
        "description": "昨夜2時ごろから、本番環境の連携で断続的に500エラーが返されるようになりました。APIコールの約15%が失敗しています。当社側では何も変更していません。エラーの発生は不規則で、同じリクエストでも再試行すると成功する場合があります。シンガポールのチームの作業が止まっているため、至急の解決をお願いいたします。",
        "status": "open"
    },
}

KB_ARTICLES = {
    "KB-001": {"title": "Processing Duplicate Payment Refunds", "content": "For duplicate charges: 1) Verify the duplicate in the billing system, 2) Issue refund through the payment processor (takes 3-5 business days), 3) Send confirmation email with refund reference number. Escalate if amount exceeds $10,000."},
    "KB-002": {"title": "Webhook Authentication After Key Rotation", "content": "When API keys are rotated, webhook signing secrets must also be updated. Go to Settings > Webhooks > Edit endpoint, and regenerate the signing secret. The old secret is invalidated immediately on key rotation. Common mistake: rotating the API key but not the webhook signing secret."},
    "KB-003": {"title": "Bulk Export Feature (Roadmap)", "content": "Bulk export is on the Q3 roadmap. Workaround: Use the REST API's /reports/export endpoint with date range parameters to programmatically export multiple reports. See API docs for batch export examples."},
    "KB-004": {"title": "Admin Account Lockout Recovery", "content": "For locked admin accounts: 1) Verify identity through the secondary email on file, 2) Reset MFA through the admin recovery flow at /admin/recover, 3) Temporary access can be granted through support-level override (requires manager approval). Critical: If SSO is blocked, enable the bypass login at /login/direct for affected users."},
    "KB-005": {"title": "API Rate Limiting Best Practices", "content": "Default rate limits: 100 requests/minute for standard plans, 1000/minute for enterprise. Use exponential backoff with jitter for retries. Monitor usage via the X-RateLimit headers in responses."},
    "KB-006": {"title": "Invoice Discrepancy Resolution", "content": "For billing discrepancies: Check the billing audit log for the account, compare with payment processor records, and verify no pending transactions. Contact finance team for adjustments over $5,000."},
    "KB-007": {"title": "Intermittent 500 Errors Troubleshooting", "content": "For intermittent server errors: 1) Check the status page for known outages, 2) Review rate limit headers - 429s can masquerade as 500s behind load balancers, 3) Check if errors correlate with payload size or specific endpoints, 4) Enable request ID logging and contact support with specific request IDs for investigation. If >10% error rate persists for >1 hour, escalate to engineering."},
}

def get_ticket(ticket_id: str) -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        return json.dumps(ticket)
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

def search_kb(query: str) -> str:
    query_lower = query.lower()
    results = []
    for article_id, article in KB_ARTICLES.items():
        if any(word in article["title"].lower() or word in article["content"].lower()
               for word in query_lower.split() if len(word) > 2):
            results.append({"id": article_id, **article})
    if not results:
        results = [{"id": "KB-000", "title": "No matches found", "content": "No relevant articles found. Consider escalating to Tier 2 support."}]
    return json.dumps(results[:3])

def resolve_ticket(ticket_id: str, resolution: str, status: str = "resolved") -> str:
    ticket = TICKETS.get(ticket_id)
    if ticket:
        ticket["status"] = status
        ticket["resolution"] = resolution
        return json.dumps({"success": True, "ticket_id": ticket_id, "new_status": status})
    return json.dumps({"error": f"Ticket {ticket_id} not found"})

TOOL_FUNCTIONS = {"get_ticket": get_ticket, "search_kb": search_kb, "resolve_ticket": resolve_ticket}

def execute_tool(name: str, input_data: dict) -> str:
    func = TOOL_FUNCTIONS.get(name)
    if func:
        return func(**input_data)
    return json.dumps({"error": f"Unknown tool: {name}"})

print("モックツールとサンプルデータを読み込みました。")
print(f"   Available tickets: {', '.join(TICKETS.keys())}")
print(f"   Knowledge base articles: {len(KB_ARTICLES)}")

---
# パート1：マルチツールのエージェントループ

TechFlowのTier 1サポートチームは現在、すべてのチケットを手作業で処理しています。顧客情報の確認、ナレッジベースの検索、問題の分類、解決内容の下書きという流れです。1日に500件以上、1件あたり約8分を要しています。TechFlowは、この作業をClaudeに自律的に任せたいと考えています。

本パートでは、この業務フローを置き換えるエージェントを構築します。構成はツール3つとループ1つで、フレームワークは使用しません。

> **重要な概念：** Claude Sonnet 5は*アダプティブ思考*に対応しており、タスクの複雑さに応じて推論の量を自動で判断します。すべてのAPI呼び出しで `thinking={"type": "adaptive"}` を指定して有効にします。

## パート2：ツールスキーマの定義

TechFlowのサポートエージェントには、3つのシステムへのアクセスが必要です。チケット管理プラットフォーム（詳細の参照）、ナレッジベース（解決策の検索）、解決エンジン（チケットのクローズ）です。それぞれをツールスキーマとして定義し、利用できるツールとその呼び出し方をClaudeに伝えます。

`description` フィールドは特に重要です。Claudeは、各ステップで*どの*ツールを使うかを、この説明から判断します。説明があいまいな場合、誤ったツールが選択されます。

> 📖 **参考：** [Tool use documentation](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview)

### ✏️ 実装課題：`search_kb` と `resolve_ticket` のツールスキーマの完成

`get_ticket` は参考例として記入済みです。`<-- fill this in` と記載された箇所で、空の `description` 文字列と `status` のenumを記入してください。

In [ ]:
# TODO: search_kb と resolve_ticket のツールスキーマを定義する
# 各ツールに必要なもの：name、description、input_schema（properties と required 付き）

tools = [
    # ✅ 例：get_ticket は完成済み。残りの2つの参考にしてください
    {
        "name": "get_ticket",
        "description": "サポートチケットのIDを指定し、顧客、優先度、製品領域、説明を含む詳細情報をすべて取得する。",
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string", "description": "チケットID（例：TKT-1042）"}
            },
            "required": ["ticket_id"]
        }
    },

    # ✏️ 実装課題：search_kb を記入する
    # ヒント：Claudeは description を見て、このツールを呼び出すタイミングを判断する。具体的に記述すること
    {
        "name": "search_kb",
        "description": "",  # <-- fill this in（ここに記入）
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": ""}  # <-- description を記入
            },
            "required": ["query"]
        }
    },

    # ✏️ 実装課題：resolve_ticket を記入する
    # ヒント：status は enum にする。解決状態として考えられる3つの値は何か
    {
        "name": "resolve_ticket",
        "description": "",  # <-- fill this in（ここに記入）
        "input_schema": {
            "type": "object",
            "properties": {
                "ticket_id": {"type": "string"},
                "resolution": {"type": "string"},
                "status": {"type": "string", "enum": []}  # <-- enumの値を記入
            },
            "required": ["ticket_id", "resolution", "status"]
        }
    }
]

print(f"Defined {len(tools)} tool schemas: {[t['name'] for t in tools]}")

## パート3：エージェントループの構築

ここが自動化の中核です。チケットが届くと、Claudeがチケットを参照し、解決策を検索し、チケットを解決します。この一連の処理を、人手を介さずに実行します。これを可能にするのがエージェントループです。`while response.stop_reason == "tool_use"` の間、ツール呼び出しを取り出して実行し、結果を追加してから、再度APIを呼び出します。処理の順序はClaudeが判断し、コードはその進行を制御するだけです。

**ポイント：** `response.content` は加工せず、そのまま messages に追加してAPIに渡してください。tool_useブロックと一緒に、thinkingブロックが含まれている場合があります。Claude Sonnet 5では、すべての呼び出しで `thinking={"type": "adaptive"}` を使用します。

> 📖 **参考：** [Agentic tool use patterns](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/agentic-tool-use)

### ✏️ 実装課題：エージェントループの実装

下の `run_agent()` にある `___` の空欄をすべて埋めてください。ループ条件、ツール結果の受け渡し、後続のAPI呼び出しの3か所です。

In [ ]:
SYSTEM_PROMPT = """あなたは、TechFlowのTier 1サポートエージェントです。TechFlowは、中堅企業向けにプロジェクト管理とチームコラボレーションのツールを提供するB2B SaaSプラットフォームです。

## 役割
受信したサポートチケットについて、問題を調査し、ナレッジベースから解決策を見つけ、明確で実行可能なガイダンスを添えてチケットを解決します。

## プロセス
1. 必ず最初にチケットを参照し、状況の全体像を把握すること
2. ナレッジベースを検索し、関連する解決策と手順を確認すること
3. 具体的な次のステップを含む詳細な解決内容で、チケットを解決すること

## ガイドライン
- 徹底する：問題が単純に見える場合でも、解決する前に必ずKBを検索する
- 具体的に記述する：解決内容には、正確な手順、リンク、所要時間を含める
- 必要に応じてエスカレーションする：確信度が低い場合や、特権アクセスが必要な問題の場合は、エスカレーション対象としてマークする
- 正確に分類する：billing、technical、account、feature_requestのいずれか
- ナレッジベースの記事は英語で書かれているため、search_kbのクエリには英語のキーワードを使用する

## エスカレーション基準
- $10,000を超える金銭に関する問題
- セキュリティに関わるアカウントの侵害
- エンジニアリングの介入が必要な問題
- Enterprise SLAのお客様（1時間以内に応答）

## TechFlowの製品ティア
- Starter（$29/ユーザー/月）：基本的なプロジェクト管理、5GBのストレージ、メールサポート、最大5プロジェクト、コミュニティフォーラム
- Professional（$79/ユーザー/月）：高度な分析、100GBのストレージ、優先サポート、APIアクセス、プロジェクト数無制限、カスタムフィールド、ガントチャート、時間トラッキング
- Enterprise（個別見積り）：SSO/SAML、ストレージ無制限、専任CSM、カスタム連携、SLA保証、監査ログ、高度なセキュリティ、カスタムブランディング、優先的なAPIレート制限

## 主な問題カテゴリとルーティング
- Billing（請求）：請求書の不一致、支払いの失敗、プランの変更、返金リクエスト、サブスクリプションの解約、日割り計算に関する質問
- Technical（技術）：APIエラー、連携の問題、Webhookの失敗、パフォーマンスの問題、データエクスポートの問題、ブラウザ互換性
- Account（アカウント）：ログインの問題、MFAの問題、SSOの設定、権限の変更、チーム管理、ユーザープロビジョニング
- Feature Requests（機能リクエスト）：製品へのフィードバック、ロードマップの問い合わせ、回避策のリクエスト、ベータ版アクセスのリクエスト

## 回答テンプレート
請求の問題を解決するときは、必ず次を含めること：取引ID、返金の所要期間、確認メールの詳細。
技術的な問題を解決するときは、必ず次を含めること：再現手順、回避策（ある場合）、エスカレーションした場合はエンジニアリングのチケット番号。
アカウントの問題を解決するときは、必ず次を含めること：実施したセキュリティ確認の手順、付与した一時的なアクセス権（ある場合）。

## SLA要件
- Starter：24時間以内に応答、営業時間内のみ
- Professional：4時間以内に応答、延長時間帯（6:00〜22:00）
- Enterprise：1時間以内に応答、24/7サポート、専用のSlackチャンネル

## トーン
プロフェッショナルかつ共感的で、解決志向の対応とする。解決策を示す前に、お客様の不満を受け止める。お客様名が分かる場合は名前を使う。ガイダンスでは、該当する製品ティアに言及する。"""


# TODO: run_agent(user_message) を実装する
# 1. ユーザーメッセージを含む messages リストを作成する
# 2. client.messages.create() を次の引数で呼び出す：
#    - model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT, tools=tools
#    - thinking={"type": "adaptive"}
#    - messages=messages
# 3. response.stop_reason == "tool_use" の間、次を繰り返す：
#    a. response.content をたどり、tool_use ブロックを見つける
#    b. execute_tool(block.name, block.input) で各ツールを実行する
#    c. tool_use_id と content を持つ tool_result の dict を作る
#    d. アシスタントの応答とツール結果を messages に追加する
#       （thinking ブロックを含め、すべてのコンテンツブロックを渡すこと）
#    e. もう一度APIを呼び出す
# 4. 最終的なresponseを返す

def run_agent(user_message: str):
    """サポートチケットエージェントを実行する。"""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model=MODEL,
        max_tokens=32000,
        system=SYSTEM_PROMPT,
        tools=tools,
        thinking={"type": "adaptive"},
        messages=messages
    )

    # ✏️ 実装課題：下の ___ をすべて埋める（Claudeがツールを使い続ける間、ループを回す）
    while response.stop_reason == ___:  # <-- 「Claudeがツールを呼び出したい」ことを示す stop_reason は？

        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": ___,  # <-- この結果をツール呼び出しに対応づける、block のフィールドは？
                    "content": str(result)
                })

        # TODO: アシスタントのターンとツール結果を messages に追加する
        # ポイント：text だけでなく、thinking ブロックを含むすべてのコンテンツブロックを渡す
        messages.append({"role": "assistant", "content": ___})  # <-- what goes here?（ここに入るものは？）
        messages.append({"role": "user", "content": tool_results})

        # TODO: 更新した messages で、もう一度APIを呼び出す
        response = client.messages.create(
            ___  # <-- 最初の create() と同じパラメータ
        )

    return response


# 動作確認
# response = run_agent("チケットTKT-1042を解決してください")
# for block in response.content:
#     if block.type == "text" and block.text.strip():
#         print(f"\n 最終応答：\n{block.text}")

## パート4：構造化出力の追加

TechFlowの下流システムには、機械可読な形式の解決結果が必要です。チケット管理プラットフォームはデータベースを更新し、分析ダッシュボードはカテゴリを集計し、エスカレーションのルーターは `escalation_needed` フラグを確認します。自由形式のテキストでは、これらの要件を満たせません。

`output_config.format` は、Claudeのテキスト応答をJSONスキーマに準拠させる制約です。**重要：** この形式の制約は*すべての*テキスト出力に適用されるため、ツールループの完了後に行う最終API呼び出しにのみ追加します。ループの間、Claudeは形式の制約を受けずに、通常どおりツールを使用します。ツールの使用が終わったら、`output_config.format` と `tool_choice={"type": "none"}` を指定してもう一度呼び出し、構造化されたJSONの解決結果を取得します。

**注意：** アダプティブ思考を使用すると、レスポンスに `[thinking, text]` のブロックが含まれる場合があります。構造化JSONは*最後*のtextブロックに含まれます。

> 📖 **参考：** [Structured outputs / JSON mode](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)

### ✏️ 実装課題：`run_agent_structured()` の完成

ツールループは記述済みです。最終呼び出しにある2つの `___`（`output_config` と `tool_choice`）を埋めてください。

In [ ]:
# ✅ RESOLUTION_SCHEMA は下で定義済み。内容を確認してから、run_agent_structured() を実装してください
RESOLUTION_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "diagnosis": {"type": "string", "description": "問題の根本原因の分析"},
            "solution_steps": {"type": "array", "items": {"type": "string"}, "description": "解決のための手順（実行順）"},
            "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
            "escalation_needed": {"type": "boolean"},
            "category": {"type": "string", "enum": ["billing", "technical", "account", "feature_request"]}
        },
        "required": ["diagnosis", "solution_steps", "confidence", "escalation_needed", "category"],
        "additionalProperties": False
    }
}


def get_structured_result(response) -> dict:
    """レスポンスの最後のtextブロックから、構造化JSONを取り出す。"""
    # アダプティブ思考では、content が [thinking, text] になることがある。JSONは最後の text ブロックにある
    text_blocks = [b for b in response.content if b.type == "text" and b.text.strip()]
    if text_blocks:
        return json.loads(text_blocks[-1].text)
    return None


def run_agent_structured(user_message: str) -> dict:
    """構造化JSON出力でエージェントを実行する。"""
    # ステップ1：ツールループを実行する。run_agent() と同じで、ここでは output_config を指定しない
    # （format はすべてのテキスト出力を制約するため、有効にするとツールが機能しない）
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(
        model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
        tools=tools, thinking={"type": "adaptive"}, messages=messages
    )
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = execute_tool(block.name, block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(
            model=MODEL, max_tokens=32000, system=SYSTEM_PROMPT,
            tools=tools, thinking={"type": "adaptive"}, messages=messages
        )

    # ステップ2：構造化出力を得るため、最後にもう一度呼び出す
    messages.append({"role": "user", "content": "構造化された解決結果をJSONで出力してください。"})
    # ✏️ 実装課題：下の2つの ___ を埋める
    final = client.messages.create(
        model=MODEL,
        max_tokens=8000,
        system=SYSTEM_PROMPT,
        output_config=___,   # <-- ここで {"format": RESOLUTION_SCHEMA} の形で RESOLUTION_SCHEMA を渡す
        tool_choice=___,     # <-- 以降のツール呼び出しを止める：{"type": "none"}
        thinking={"type": "adaptive"},
        messages=messages
    )
    return get_structured_result(final)


# result = run_agent_structured("チケットTKT-1042を解決してください")
# print(json.dumps(result, indent=2))

---
### ✅ チェックポイント1：構造化出力を備えたサポートチケットエージェント

チケット1件につき2〜3個のツールを呼び出し、構造化JSONを返すエージェントが完成しているはずです。

**確認：** `run_agent_structured("Resolve ticket TKT-1044")`（日本語版では `run_agent_structured("チケットTKT-1044を解決してください")`）を実行し、`category` が `feature_request` となり、APIを使った回避策が提案されることをご確認ください。

---
# パート2：アダプティブ思考

基本のエージェントは動作していますが、TechFlowのサポートリーダーから次の懸念が示されました。「二重請求やパスワードのリセットのように単純なチケットもあれば、判断の難しいチケットもあります。難しいチケットでは深く推論させたい一方で、簡単なチケットの処理速度は落としたくありません。また、難しいチケットで判断を誤った場合に、*なぜ*その判断に至ったのかを確認できない状況です。」

アダプティブ思考は、この両方の課題を解決します。`output_config.effort` パラメータで、Claudeの推論の深さを制御できます。複雑なチケットやあいまいなチケットにはhigh、単純なチケットにはlowを指定します。思考トレースによって判断のプロセスが可視化されるため、エージェントが*なぜ*エスカレーションを選んだのか、どのKB記事を最も重視したのかを監査できます。

> 📖 **参考：** [Adaptive thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)

## セル5：エフォートレベルによる思考制御の追加

`output_config.effort` を追加し、Claudeの推論の深さを制御します。highエフォートでは、ツール呼び出しの合間に詳細な思考トレースが生成されます。何を検索するか、KBの結果をどう評価するか、エスカレーションすべきかを推論します。lowエフォートでは、単純なチケット向けに推論を短く抑えます。思考ブロックを表示し、エージェントの判断プロセスを確認してください。

### ✏️ 実装課題：`run_agent_thinking()` の実装

今回は関数全体を記述します。コメントに沿って、4つのステップを進めてください。

In [ ]:
# TODO: エージェントに、エフォートレベルによる思考制御を追加する
# 1. run_agent をコピーしてツールループを実行する。その際、thinking={"type": "adaptive"}
#    と output_config={"effort": effort} を渡す（format は渡さない。最終呼び出しで使用する）
# 2. ループの中で、思考ブロックを表示する：block.type == "thinking"
# 3. ツールループの終了後、次の指定で最終呼び出しを行う：
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
#    - 「構造化された解決結果をJSONで出力してください。」のようなユーザーメッセージを追加する
# 4. get_structured_result() を使って、最終レスポンスを解析する

def run_agent_thinking(user_message: str, effort: str = "high") -> dict:
    """エフォートレベルで制御するアダプティブ思考を有効にして、エージェントを実行する。"""
    # ✏️ YOUR TURN（実装課題）：ここに実装を記述する
    pass

## セル6：アダプティブ思考の動作確認

MedTech Solutions（TKT-1046）から、断続的な500エラーが報告されています。APIコールの15%が失敗しており、先方での変更はなく、2時から発生しています。レート制限か、サーバー側の障害か、ネットワークの問題か。KBに完全に一致する記事はありません。

思考が真価を発揮するのは、このような場面です。判断の難しいチケットをhighエフォートで実行し、同じチケットをlowエフォートで実行して比較してください。推論の深さがどう変わるかを観察したうえで、次の点をご検討ください。TechFlowの1日500件のチケットにおいて、単純なチケットと複雑なチケットを、それぞれどのエフォートレベルに振り分けるべきでしょうか？

In [ ]:
def _not_built_yet(fn_name, result):
    """ハンズオン用のガード：実装課題（# YOUR TURN）のスタブは、実装するまで None を返す。"""
    if result is not None:
        return False
    print(f"\n[--] {fn_name}() isn't built yet - that's the exercise, not a bug.")
    print(f"     Find the '# YOUR TURN' marker inside {fn_name}(), build it, then run this again.")
    return True

# あいまいなチケットをhighエフォートで実行し、思考トレースを観察する
print("=== TKT-1046：断続的なAPIエラー（判断の難しいチケット） ===\n")
result = run_agent_thinking("チケットTKT-1046を解決してください", effort="high")
if _not_built_yet("run_agent_thinking", result):
    raise SystemExit(0)
print(f"\nResolution:")
print(json.dumps(result, indent=2))

# 次に比較：同じチケットをlowエフォートで実行する
print(f"\n\n{'='*50}")
print("=== 同じチケット、LOWエフォート ===")
print(f"{'='*50}\n")

for effort in ["high", "low"]:
    start = time.time()
    result = run_agent_thinking("チケットTKT-1046を解決してください", effort=effort)
    elapsed = time.time() - start
    print(f"\n[effort={effort}] Confidence: {result['confidence']} | Steps: {len(result['solution_steps'])} | Escalate: {result['escalation_needed']} | Time: {elapsed:.1f}s")

---
### ✅ チェックポイント2：推論の可視化とエフォート制御を備えたエージェント

エフォートレベルを比較すると、推論の深さと質に明確な違いが確認できるはずです。

---
# パート3：ストリーミング

TechFlowのサポート担当者は、リアルタイムのダッシュボードでAIによるチケット振り分けを監視しています。チケットが届いた際、担当者はエージェントの作業状況を*確認*できる必要があります。どのチケットを参照しているのか、何を検索しているのか、どのように推論しているのか。15秒間スピナーが回った後に長文が一度に表示されるだけでは、信頼は得られません。

この課題を解決するのがストリーミングです。`create()` を `stream()` に置き換えると、トークンがリアルタイムで届きます。エージェントの推論中は思考トレースが流れ、ツール呼び出しは実行されるたびに表示され、構造化された解決結果は最後にストリーミングされます。

> 📖 **参考：** [Streaming messages](https://docs.anthropic.com/en/docs/build-with-claude/streaming)

## セル7：ストリーミング対応のエージェントループ

`client.messages.create()` を `client.messages.stream()` に置き換え、TechFlowのダッシュボードにエージェントの作業をリアルタイムで表示できるようにします。主要なイベントタイプとして、`thinking_delta`（推論のトークン）、`text_delta`（最終応答）、`input_json_delta`（ツールの引数）を処理します。

ストリームの完了後は、`stream.get_final_message()` でレスポンスオブジェクト全体を取得し、ループの継続に使用します。

### ✏️ 実装課題：`run_agent_streaming()` の実装

関数全体を記述します。`create()` を `stream()` に置き換え、コメントに記載したストリームイベントを処理してください。

In [ ]:
# TODO: ストリーミング対応のエージェントループを構築する
# 1. create() を stream() に置き換え、コンテキストマネージャ（with ... as stream:）を使う
#    ツールループの間は output_config={"effort": effort} を使う（formatの制約はなし）
# 2. ストリームイベントを順に処理し、次のイベントを扱う：
#    - content_block_start：content_block.type を確認する（thinking/tool_use/text）
#    - content_block_delta：thinking_delta、text_delta、input_json_deltaを処理する
# 3. ストリーミングの後、stream.get_final_message() で完全なレスポンスを取得する
# 4. stop_reason が tool_use なら、ツールを実行してループを続ける
# 5. ツールループの終了後、次の指定でストリーミングの最終呼び出しを行う：
#    - output_config={"effort": effort, "format": RESOLUTION_SCHEMA}
#    - tool_choice={"type": "none"}
# 6. 最終的なJSONには、get_structured_result() を使う
# 注意：stream()に thinking={"type": "adaptive", "display": "summarized"} を渡すこと

def run_agent_streaming(user_message: str, effort: str = "high") -> dict:
    """ストリーミング出力でエージェントを実行する。"""
    # ✏️ YOUR TURN（実装課題）：ここに実装を記述する
    pass

## セル8：フルデモ

SecureNet Systemsから重大なチケットが届きました。管理者アカウントがロックアウトされ、47名のチームメンバーが作業できない状態です。ストリーミングを有効にしてエージェント全体を実行し、4つの機能がリアルタイムで連携する様子を確認してください。エージェントループがツール呼び出しを制御し、構造化出力が解決結果の形式を保証し、アダプティブ思考がセキュリティ上の影響を推論し、ストリーミングがその過程をすべてリアルタイムで表示します。

In [ ]:
print("エージェントのフルデモ：TKT-1045の解決（アカウントのロックアウト）")
print("   ストリーミング + アダプティブ思考 + ツール + 構造化出力")
print("=" * 60)

start = time.time()
result = run_agent_streaming("チケットTKT-1045を解決してください")
elapsed = time.time() - start
if _not_built_yet("run_agent_streaming", result):
    raise SystemExit(0)

print(f"\n\n{'=' * 60}")
print(f"Total time: {elapsed:.1f}s")
print(f"\nStructured Resolution:")
print(json.dumps(result, indent=2))

---
### ✅ チェックポイント3：リアルタイムで動作するエージェント

1回のエージェント実行の中で、4つの機能が連携しています。
エージェントループ＋構造化出力＋アダプティブ思考＋ストリーミング

---
# 発展課題

1. **ツール選択の制御**：`tool_choice: {"type": "none"}` を使用し、Claudeのツール呼び出しを停止させる
2. **エフォートの最適化**：すべてのチケットを `effort="high"`、`effort="medium"`、`effort="low"` で処理し、品質と速度の比較表を作成する
3. **バッチ処理**：Batch APIを使用して、すべてのチケットを一括処理する（コストを50%削減）

## 参考資料

- [Claude API Documentation](https://docs.anthropic.com/en/docs)
- [Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [Adaptive Thinking](https://docs.anthropic.com/en/docs/build-with-claude/extended-thinking)
- [Structured Outputs](https://docs.anthropic.com/en/docs/build-with-claude/structured-outputs)
- [Streaming](https://docs.anthropic.com/en/docs/build-with-claude/streaming)